# Issue with CMIP6 cesm2 psl dataset

When: 2026-02-02
Versions: clisops = v0.17.0 , rook = v0.18.0

There has been an error reported with the CMIP6 entry, which the CDS team quickly tested, here is the summary:

All requests I tested fail for the variable Sea level pressure and model CESM2 (USA).  The error says

`Process error: Resulting object does not have monotonic global indexes along dimension time.`

Tests for other random variables and models were fine.

I guess this is a Roocs error triggered by time subsetting, due to an issue with the time dimension in the SLP file from this model?

Here is a failing request:
```
{"inputs": {"psl": ["c3s-cmip6.CMIP.NCAR.CESM2.historical.r1i1p1f1.day.psl.gn.v20190308"]}, "steps": {"subset_psl_1": {"run": "subset", "in": {"collection": "inputs/psl", "time_components": "day:01|month:jan|year:1900", "time": "1900/1900"}}}, "outputs": {"output": "subset_psl_1/output"}, "doc": "workflow"}
```

Exception in log:
```
/usr/local/anaconda/envs/rook/lib/python3.12/site-packages/xarray/conventions.py:204: SerializationWarning: variable 'psl' has multiple fill values {1e+20, 1e+20} defined, decoding all values to NaN.
  var = coder.decode(var, name=name)
/usr/local/anaconda/envs/rook/lib/python3.12/site-packages/clisops/utils/dataset_utils.py:448: FutureWarning: In a future version of xarray the default value for data_vars will change from data_vars='all' to data_vars=None. This is likely to lead to different results when multiple datasets have matching variables with overlapping values. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set data_vars explicitly.
  ds = xr.open_mfdataset(dset, **multi_file_kwargs)
Traceback (most recent call last):
  File "/usr/local/anaconda/envs/rook/lib/python3.12/site-packages/rook/director/director.py", line 204, in process
    file_uris = runner(self.inputs)
                ^^^^^^^^^^^^^^^^^^^
  File "/usr/local/anaconda/envs/rook/lib/python3.12/site-packages/rook/utils/subset_utils.py", line 69, in run_subset
    result = subset(**args)
             ^^^^^^^^^^^^^^
  File "/usr/local/anaconda/envs/rook/lib/python3.12/site-packages/rook/utils/subset_utils.py", line 61, in subset
    result_set = Subset(**locals()).calculate()
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/anaconda/envs/rook/lib/python3.12/site-packages/rook/utils/subset_utils.py", line 35, in calculate
    norm_collection = normalise.normalise(self.collection, False)
                      ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/anaconda/envs/rook/lib/python3.12/site-packages/daops/utils/normalise.py", line 22, in normalise
    ds = open_dataset(dset, file_paths, apply_fixes)
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/anaconda/envs/rook/lib/python3.12/site-packages/daops/utils/core.py", line 94, in open_dataset
    ds = open_xr_dataset(file_paths)
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/anaconda/envs/rook/lib/python3.12/site-packages/clisops/utils/dataset_utils.py", line 448, in open_xr_dataset
    ds = xr.open_mfdataset(dset, **multi_file_kwargs)
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/anaconda/envs/rook/lib/python3.12/site-packages/xarray/backends/api.py", line 1669, in open_mfdataset
    combined = combine_by_coords(
               ^^^^^^^^^^^^^^^^^^
  File "/usr/local/anaconda/envs/rook/lib/python3.12/site-packages/xarray/structure/combine.py", line 1035, in combine_by_coords
    concatenated_grouped_by_data_vars = tuple(
                                        ^^^^^^
  File "/usr/local/anaconda/envs/rook/lib/python3.12/site-packages/xarray/structure/combine.py", line 1036, in <genexpr>
    _combine_single_variable_hypercube(
  File "/usr/local/anaconda/envs/rook/lib/python3.12/site-packages/xarray/structure/combine.py", line 720, in _combine_single_variable_hypercube
    raise ValueError(
ValueError: Resulting object does not have monotonic global indexes along dimension time
```


## Error analysis by ChatGPT

### Issue
`xarray.open_mfdataset` fails with  
`ValueError: Resulting object does not have monotonic global indexes along dimension time`

### Description
When Rook / clisops opens multiple NetCDF files using `xr.open_mfdataset(combine="by_coords")`, xarray requires the `time` coordinate to be globally monotonic across all input files. This error indicates that the combined datasets contain non-monotonic, overlapping, duplicated, or out-of-order `time` values.

### Potential causes
- Input files are not ordered by time  
- Overlapping or duplicate time ranges between files  
- Inconsistent time encodings or calendars  

### Potential fixes
- Sort files by time before opening  
- Sort `time` within each dataset using `preprocess=lambda ds: ds.sortby("time")`  
- Detect and remove overlapping or duplicate timestamps  
- As a last resort, use `combine="nested", concat_dim="time"` if file ordering is guaranteed


## Search Catalog

In [1]:
import intake

In [2]:
cat_url = "https://raw.githubusercontent.com/cp4cds/c3s_34g_manifests/master/intake/catalogs/c3s.yaml"

cat = intake.open_catalog(cat_url)
list(cat)

['c3s-cmip5',
 'c3s-cmip5-daily-pressure-level',
 'c3s-cmip5-daily-single-level',
 'c3s-cmip5-monthly-pressure-level',
 'c3s-cmip5-monthly-single-level',
 'c3s-cmip6',
 'c3s-cmip6-decadal',
 'c3s-cordex',
 'c3s-ipcc-atlas',
 'c3s-cica-atlas-v01',
 'c3s-cica-atlas']

In [3]:
df_cmip6 = cat['c3s-cmip6'].read()
df_cmip6.head()

,ds_id,path,size,mip_era,activity_id,institution_id,source_id,experiment_id,member_id,table_id,variable_id,grid_label,version,start_time,end_time,bbox,level
0,c3s-cmip6.ScenarioMIP.MOHC.UKESM1-0-LL.ssp245....,ScenarioMIP/MOHC/UKESM1-0-LL/ssp245/r1i1p1f2/A...,28037112,c3s-cmip6,ScenarioMIP,MOHC,UKESM1-0-LL,ssp245,r1i1p1f2,Amon,ts,gn,v20190507,2015-01-16T00:00:00,2049-12-16T00:00:00,"0.94, -89.38, 359.06, 89.38",NaN
1,c3s-cmip6.ScenarioMIP.MOHC.UKESM1-0-LL.ssp245....,ScenarioMIP/MOHC/UKESM1-0-LL/ssp245/r1i1p1f2/A...,38838222,c3s-cmip6,ScenarioMIP,MOHC,UKESM1-0-LL,ssp245,r1i1p1f2,Amon,ts,gn,v20190507,2050-01-16T00:00:00,2100-12-16T00:00:00,"0.94, -89.38, 359.06, 89.38",NaN
2,c3s-cmip6.ScenarioMIP.NCAR.CESM2.ssp370.r4i1p1...,ScenarioMIP/NCAR/CESM2/ssp370/r4i1p1f1/Amon/pr...,104081588,c3s-cmip6,ScenarioMIP,NCAR,CESM2,ssp370,r4i1p1f1,Amon,pr,gn,v20200528,2015-01-15T12:00:00,2064-12-15T12:00:00,"0.00, -90.00, 358.75, 90.00",NaN
3,c3s-cmip6.ScenarioMIP.NCAR.CESM2.ssp370.r4i1p1...,ScenarioMIP/NCAR/CESM2/ssp370/r4i1p1f1/Amon/pr...,74977662,c3s-cmip6,ScenarioMIP,NCAR,CESM2,ssp370,r4i1p1f1,Amon,pr,gn,v20200528,2065-01-15T12:00:00,2100-12-15T12:00:00,"0.00, -90.00, 358.75, 90.00",NaN
4,c3s-cmip6.ScenarioMIP.AS-RCEC.TaiESM1.ssp370.r...,ScenarioMIP/AS-RCEC/TaiESM1/ssp370/r1i1p1f1/Am...,144277888,c3s-cmip6,ScenarioMIP,AS-RCEC,TaiESM1,ssp370,r1i1p1f1,Amon,rlut,gn,v20201014,2015-01-16T12:00:00,2100-12-16T12:00:00,"0.00, -90.00, 358.75, 90.00",NaN


In [4]:
df = df_cmip6.loc[
    (df_cmip6.source_id=="CESM2" )
    & (df_cmip6.table_id=="day")
    & (df_cmip6.variable_id=="psl")
    & (df_cmip6.experiment_id=="historical")
]
df

,ds_id,path,size,mip_era,activity_id,institution_id,source_id,experiment_id,member_id,table_id,variable_id,grid_label,version,start_time,end_time,bbox,level
62617,c3s-cmip6.CMIP.NCAR.CESM2.historical.r1i1p1f1....,CMIP/NCAR/CESM2/historical/r1i1p1f1/day/psl/gn...,7118009724,c3s-cmip6,CMIP,NCAR,CESM2,historical,r1i1p1f1,day,psl,gn,v20190308,1850-01-01T00:00:00,2015-01-01T00:00:00,"0.00, -90.00, 358.75, 90.00",NaN
62625,c3s-cmip6.CMIP.NCAR.CESM2.historical.r1i1p1f1....,CMIP/NCAR/CESM2/historical/r1i1p1f1/day/psl/gn...,431515502,c3s-cmip6,CMIP,NCAR,CESM2,historical,r1i1p1f1,day,psl,gn,v20190308,1860-01-01T00:00:00,1869-12-31T00:00:00,"0.00, -90.00, 358.75, 90.00",NaN
62633,c3s-cmip6.CMIP.NCAR.CESM2.historical.r1i1p1f1....,CMIP/NCAR/CESM2/historical/r1i1p1f1/day/psl/gn...,431429952,c3s-cmip6,CMIP,NCAR,CESM2,historical,r1i1p1f1,day,psl,gn,v20190308,1870-01-01T00:00:00,1879-12-31T00:00:00,"0.00, -90.00, 358.75, 90.00",NaN
62641,c3s-cmip6.CMIP.NCAR.CESM2.historical.r1i1p1f1....,CMIP/NCAR/CESM2/historical/r1i1p1f1/day/psl/gn...,431367520,c3s-cmip6,CMIP,NCAR,CESM2,historical,r1i1p1f1,day,psl,gn,v20190308,1880-01-01T00:00:00,1889-12-31T00:00:00,"0.00, -90.00, 358.75, 90.00",NaN
62649,c3s-cmip6.CMIP.NCAR.CESM2.historical.r1i1p1f1....,CMIP/NCAR/CESM2/historical/r1i1p1f1/day/psl/gn...,431336563,c3s-cmip6,CMIP,NCAR,CESM2,historical,r1i1p1f1,day,psl,gn,v20190308,1890-01-01T00:00:00,1899-12-31T00:00:00,"0.00, -90.00, 358.75, 90.00",NaN
62657,c3s-cmip6.CMIP.NCAR.CESM2.historical.r1i1p1f1....,CMIP/NCAR/CESM2/historical/r1i1p1f1/day/psl/gn...,431427018,c3s-cmip6,CMIP,NCAR,CESM2,historical,r1i1p1f1,day,psl,gn,v20190308,1900-01-01T00:00:00,1909-12-31T00:00:00,"0.00, -90.00, 358.75, 90.00",NaN
62665,c3s-cmip6.CMIP.NCAR.CESM2.historical.r1i1p1f1....,CMIP/NCAR/CESM2/historical/r1i1p1f1/day/psl/gn...,431417362,c3s-cmip6,CMIP,NCAR,CESM2,historical,r1i1p1f1,day,psl,gn,v20190308,1910-01-01T00:00:00,1919-12-31T00:00:00,"0.00, -90.00, 358.75, 90.00",NaN
62673,c3s-cmip6.CMIP.NCAR.CESM2.historical.r1i1p1f1....,CMIP/NCAR/CESM2/historical/r1i1p1f1/day/psl/gn...,431462467,c3s-cmip6,CMIP,NCAR,CESM2,historical,r1i1p1f1,day,psl,gn,v20190308,1920-01-01T00:00:00,1929-12-31T00:00:00,"0.00, -90.00, 358.75, 90.00",NaN
62681,c3s-cmip6.CMIP.NCAR.CESM2.historical.r1i1p1f1....,CMIP/NCAR/CESM2/historical/r1i1p1f1/day/psl/gn...,431403078,c3s-cmip6,CMIP,NCAR,CESM2,historical,r1i1p1f1,day,psl,gn,v20190308,1930-01-01T00:00:00,1939-12-31T00:00:00,"0.00, -90.00, 358.75, 90.00",NaN
62689,c3s-cmip6.CMIP.NCAR.CESM2.historical.r1i1p1f1....,CMIP/NCAR/CESM2/historical/r1i1p1f1/day/psl/gn...,431460703,c3s-cmip6,CMIP,NCAR,CESM2,historical,r1i1p1f1,day,psl,gn,v20190308,1940-01-01T00:00:00,1949-12-31T00:00:00,"0.00, -90.00, 358.75, 90.00",NaN


In [5]:
ds_ids = list(df.ds_id.unique())
ds_ids

['c3s-cmip6.CMIP.NCAR.CESM2.historical.r1i1p1f1.day.psl.gn.v20190308']

## Run subset operator

In [6]:
import os
os.environ['ROOK_URL'] = 'http://rook.dkrz.de/wps'

from rooki import rooki

### Failed Request

In [7]:
resp = rooki.subset(
    collection=ds_ids,
    time='1900/1900',
    time_components='day:01|month:jan|year:1900',
    area='90,-180,-90,180'
)
resp.ok

False

In [8]:
resp

Process error: Resulting object does not have monotonic global indexes along dimension time